# Understanding vectorizers

In the following code examples, we will experiment with vectorizers to understand a bit better how they work. Feel free to adjust the code, and try things out yourself.

For now, we will practice with `sklearn`'s vectorizers. However, packages such as `gensim` offer their own built-in functionality to vectorize the data. 
You can also play around [here](https://apps-computational-teaching-jj92xohbgnwnhksxlclr8t.streamlit.app/).


## Setup

Run this first if `sklearn` is not installed yet.

In [ ]:
!pip install scikit-learn pandas

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
import sklearn
print(sklearn.__version__)

1.8.0


## How to read this notebook

This notebook shows how **vectorizers** turn text into numbers.

Keep this idea in mind throughout:

- **rows** = documents  
- **columns** = vocabulary terms (tokens) 
- **values** = what the vectorizer stores for that term in that document  

For the two vectorizers in this notebook:

- **CountVectorizer** stores **raw counts**
- **TfidfVectorizer** stores **weights**


So the basic workflow is always:

1. start with text  
2. build a vocabulary  
3. turn each document into a row of numbers  
4. inspect what those numbers mean


## Example 1: Inspect the output of a vectorizer in a dense format

The next code cell fits a vectorizer on a small list of documents and then converts the result to **dense** format so it is easier to inspect.

### What should you look for?

- the number of **rows** should match the number of documents
- the number of **columns** should match the number of vocabulary terms
- the **values** tell you what the vectorizer stores

With `CountVectorizer`:

- `0` = the term does not occur
- `1` = the term occurs once
- `2` = the term occurs twice

So the output is a **document-term matrix**:
- rows = documents
- columns = words
- values = counts

Dense format is useful for learning because you can read the full matrix directly.  
In practice, vectorizers usually store the result as a **sparse matrix**, because most entries are zero.


In [3]:
texts = ["hello students!", "how are you today?", "what?", "hello hello everybody"]
vect = CountVectorizer()


In [4]:
#vect = TfidfVectorizer()# initialize the vectorizer
X = vect.fit_transform(texts) #fit the vectorizer and transform the documents in one go

In [5]:
print(pd.DataFrame(X.toarray(), columns=vect.get_feature_names_out()).to_string())
df = pd.DataFrame(X.toarray().transpose(), index = vect.get_feature_names_out())

   are  everybody  hello  how  students  today  what  you
0    0          0      1    0         1      0     0    0
1    1          0      0    1         0      1     0    1
2    0          0      0    0         0      0     1    0
3    0          1      2    0         0      0     0    0


## Example 2: Inspect the output of a vectorizer in a sparse format

Internally, `sklearn` represents the data in a *sparse* format, as this is computationally more efficient, and less memory is required.


In [6]:
texts = ["hello students!", "how are you today?", "what?", "hello hello everybody"]
count_vec = CountVectorizer() #initilize the vectorizer
count_vec_fit = count_vec.fit_transform(texts) #fit the vectorizer and transform the documents in one go

1. Inspect the shape of transformed texts. We can see that we have a 4x8 sparse matrix, meaning that we have 4 
rows (=documents) and 8 unique tokens (=words, numbers)


In [7]:
count_vec_fit

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9 stored elements and shape (4, 8)>

2. Get the feature names. This will return the tokens that are in the vocabulary of the vectorizer

In [8]:
count_vec.get_feature_names_out()

array(['are', 'everybody', 'hello', 'how', 'students', 'today', 'what',
       'you'], dtype=object)

3. Represent the token's mapping to its id values.

This is the **vocabulary dictionary** of the vectorizer.  
It tells you which column index belongs to which token.

Example:

```python
{'data': 0, 'love': 1, 'science': 2}
```

This means:

- column `0` = `data`
- column `1` = `love`
- column `2` = `science`

Important:
- the **vocabulary mapping** tells you where a token is stored
- the **document-term matrix** tells you what value is stored there


In [9]:
count_vec.vocabulary_ 

{'hello': 2,
 'students': 4,
 'how': 3,
 'are': 0,
 'you': 7,
 'today': 5,
 'what': 6,
 'everybody': 1}

4. Get sparse representation on document level

### Inspecting one document at a time

Looking at the full matrix can still feel abstract.  
So below we inspect it **row by row**.

The loop pairs:

- the original document from `texts`
- its matching row from `count_vec_fit`

That makes it easier to connect:

- the human-readable text
- the machine-readable vector

We use `.toarray()` to make each sparse row easier to read.


In [10]:
for doc, row in zip(texts, count_vec_fit):
    print(f"Document: {doc}")
    print(row.toarray())
    print()

Document: hello students!
[[0 0 1 0 1 0 0 0]]

Document: how are you today?
[[1 0 0 1 0 1 0 1]]

Document: what?
[[0 0 0 0 0 0 1 0]]

Document: hello hello everybody
[[0 1 2 0 0 0 0 0]]



### A more interpretable view: show words together with their counts

A row of numbers is still hard to interpret unless you know which column belongs to which word.

So in the next cell we combine:

- the vocabulary from `count_vec.get_feature_names_out()`
- the values from one document row

Then we print only the terms with a count greater than zero.

This is often the clearest way to show what the vectorizer is doing.


In [11]:
feature_names = count_vec.get_feature_names_out()

for doc, row in zip(texts, count_vec_fit):
    print(f"Document: {doc}")

    row_array = row.toarray()[0]

    for word, count in zip(feature_names, row_array):
        if count > 0:
            print(f"{word}: {count}")

    print()

Document: hello students!
hello: 1
students: 1

Document: how are you today?
are: 1
how: 1
today: 1
you: 1

Document: what?
what: 1

Document: hello hello everybody
everybody: 1
hello: 2



5. Get some final descriptives about the sparse matrix

## Describing sparsity

A document-term matrix is usually **sparse**, which means that most cells are zero.

Why?  
Because each document only contains a small subset of the full vocabulary.

That is why scikit-learn stores these matrices in sparse form by default.

In the next cell:

- `nnz` = number of **non-zero** entries
- `rows × columns` = total number of entries
- **sparsity** = proportion of entries that are zero


In [13]:
nonzero = count_vec_fit.nnz
print("Number of non-zero elements:", nonzero)
print("Total number of elements:", count_vec_fit.shape[0] * count_vec_fit.shape[1])

# compute the sparsity of the matrix: the proportion of zero elements in the matrix
print("Sparsity:", 1 - nonzero / (count_vec_fit.shape[0] * count_vec_fit.shape[1]))

Number of non-zero elements: 9
Total number of elements: 32
Sparsity: 0.71875
